In [1]:
!pip install gradio easyocr opencv-python tensorflow

In [2]:
import gradio as gr
import tensorflow as tf
import cv2
import numpy as np
import easyocr
import joblib

In [13]:
from skimage.feature import hog, local_binary_pattern

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
model_svm_murni = joblib.load('/content/drive/MyDrive/Machine Learning I/UAS/ML & DL/svm_model2.pkl')
model_hybrid_svm = joblib.load('/content/drive/MyDrive/Machine Learning I/UAS/ML & DL/hybrid_svm_classifier.pkl')

scaler_svm = joblib.load('/content/drive/MyDrive/Machine Learning I/UAS/ML & DL/svm_scaler2.pkl')
scaler_hybrid = joblib.load('/content/drive/MyDrive/Machine Learning I/UAS/ML & DL/hybrid_scaler.pkl')

reader = easyocr.Reader(['en'], gpu=True)

In [16]:
print("Menginisialisasi Ekstraktor Fitur MobileNetV3 (ImageNet)...")
base_extractor = tf.keras.applications.MobileNetV3Small(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
gap_layer = tf.keras.layers.GlobalAveragePooling2D()

Menginisialisasi Ekstraktor Fitur MobileNetV3 (ImageNet)...
4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [17]:
def extract_hog_lbp_features(image):
    """Pipeline khusus untuk Model 1 (SVM Murni)"""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    resized_gray = cv2.resize(gray, (64, 32), interpolation=cv2.INTER_AREA)

    # Ekstraksi HOG
    fd_hog = hog(resized_gray, orientations=9, pixels_per_cell=(8, 8),
                 cells_per_block=(2, 2), visualize=False, feature_vector=True)

    # Ekstraksi LBP
    radius = 1
    n_points = 8 * radius
    lbp_image = local_binary_pattern(resized_gray, n_points, radius, method='uniform')
    n_bins = int(lbp_image.max() + 1)
    lbp_hist, _ = np.histogram(lbp_image.ravel(), bins=n_bins, range=(0, n_bins))
    lbp_hist = lbp_hist.astype("float")
    lbp_hist /= (lbp_hist.sum() + 1e-6)

    combined_vector = np.hstack((fd_hog, lbp_hist))
    return combined_vector.reshape(1, -1)

def preprocess_for_dl(image):
    """Pipeline gambar untuk input ekstrak fitur tensor"""
    img_tensor = tf.convert_to_tensor(image, dtype=tf.float32)
    img_padded = tf.image.resize_with_pad(img_tensor, 224, 224)
    img_normalized = img_padded / 255.0
    return tf.expand_dims(img_normalized, axis=0)

In [23]:
def process_license_plate(image):
    if image is None:
        return "Error", "Error", "Tolong unggah gambar plat nomor."

    # --- JALUR 1: SVM Murni + HOG/LBP (Sudah Benar) ---
    features_svm = extract_hog_lbp_features(image)
    features_svm_scaled = scaler_svm.transform(features_svm)
    try:
        prob_svm = model_svm_murni.predict_proba(features_svm_scaled)[0][1]
    except AttributeError:
        distance = model_svm_murni.decision_function(features_svm_scaled)[0]
        prob_svm = 1 / (1 + np.exp(-distance))
    res_svm = "Unreadable" if prob_svm > 0.5 else "Readable"

    # --- JALUR 3: Hybrid MobileNetV3 + SVM (PERBAIKAN FATAL) ---
    img_dl = preprocess_for_dl(image)
    raw_cnn_features = base_extractor(img_dl, training=False)
    cnn_features = gap_layer(raw_cnn_features).numpy()
    cnn_features_scaled = scaler_hybrid.transform(cnn_features)

    # 1. Ambil keputusan kelas secara langsung menggunakan .predict() agar mematuhi class_weight='balanced'
    pred_hybrid_class = model_hybrid_svm.predict(cnn_features_scaled)[0]
    res_hybrid = "Unreadable" if pred_hybrid_class == 1 else "Readable"

    # 2. Untuk visualisasi skor di Gradio, gunakan decision_function + Sigmoid agar konsisten secara matematis
    distance_hybrid = model_hybrid_svm.decision_function(cnn_features_scaled)[0]
    prob_hybrid = 1 / (1 + np.exp(-distance_hybrid))

    # --- LOGIKA GATEKEEPER & OCR (Menggunakan Penentu dari Hybrid yang Sudah Diperbaiki) ---
    if res_hybrid == "Unreadable":
        ocr_text = "DITOLAK. Sistem mendeteksi plat rusak atau tidak layak baca."
    else:
        ocr_result = reader.readtext(image, detail=0)
        plate_text = " ".join(ocr_result).upper()
        ocr_text = plate_text if plate_text else "DITERIMA (Readable), tetapi teks gagal diekstrak oleh OCR."

    return (f"{res_svm} ({prob_svm:.2f})",
            f"{res_hybrid} ({prob_hybrid:.2f})",
            ocr_text)

In [24]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Sistem Klasifikasi Plat Nomor")

    with gr.Row():
        with gr.Column(scale=1):
            img_input = gr.Image(type="numpy", label="Unggah Gambar Plat")
            submit_btn = gr.Button("Proses Plat", variant="primary")

        with gr.Column(scale=1):
            gr.Markdown("### Analisis Sensor Gatekeeper (Probabilitas Unreadable)")
            status_svm = gr.Textbox(label="Model 1: SVM Murni (HOG/LBP)")
            status_hybrid = gr.Textbox(label="Model 3: Hybrid MobileNetV3 + SVM (Penentu)")

            gr.Markdown("### Output Sistem")
            text_output = gr.Textbox(label="Hasil Baca Teks (EasyOCR)", lines=2)

    submit_btn.click(
        fn=process_license_plate,
        inputs=img_input,
        outputs=[status_svm, status_hybrid, text_output]
    )

# Jalankan aplikasi
demo.launch(share=True, debug=True)

/tmp/ipykernel_4422/2224611276.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://897e488ecc44e25022.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 62, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://897e488ecc44e25022.gradio.live
